In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [3]:
class InteractionDataset(Dataset):
    def __init__(self, user_ids, item_ids, labels):
        self.user_ids = torch.LongTensor(user_ids)
        self.item_ids = torch.LongTensor(item_ids)
        self.labels = torch.FloatTensor(labels)

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        return self.user_ids[idx], self.item_ids[idx], self.labels[idx]

In [4]:
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)

    def forward(self, user, item):
        u = self.user_embedding(user)
        i = self.item_embedding(item)
        x = (u * i).sum(dim=1)  # dot product
        return torch.sigmoid(x)

In [5]:
class NeuralCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)

        self.mlp = nn.Sequential(
            nn.Linear(2 * embedding_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, user, item):
        u = self.user_embedding(user)
        i = self.item_embedding(item)
        x = torch.cat([u, i], dim=1)
        x = self.mlp(x)
        return torch.sigmoid(x).squeeze()


In [6]:
def train_model(model, dataloader, epochs=5, lr=0.001, device="cpu"):
    model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for user, item, label in dataloader:
            user, item, label = user.to(device), item.to(device), label.to(device)

            optimizer.zero_grad()
            pred = model(user, item)
            loss = criterion(pred, label)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [7]:
def hit_at_k(model, user_ids, item_ids, k=10, device="cpu"):
    model.eval()
    hits = 0

    with torch.no_grad():
        for user in set(user_ids):
            user_tensor = torch.LongTensor([user] * len(item_ids)).to(device)
            item_tensor = torch.LongTensor(item_ids).to(device)

            scores = model(user_tensor, item_tensor)
            _, top_k_items = torch.topk(scores, k)

            # Assume ground truth = first item (for demo)
            if 0 in top_k_items:
                hits += 1

    return hits / len(set(user_ids))

In [8]:

num_users = 100
num_items = 200
embedding_dim = 32

N = 5000
user_ids = np.random.randint(0, num_users, N)
item_ids = np.random.randint(0, num_items, N)
labels = np.random.randint(0, 2, N)

dataset = InteractionDataset(user_ids, item_ids, labels)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Train Baseline
mf_model = MatrixFactorization(num_users, num_items, embedding_dim)
train_model(mf_model, dataloader)

# Train MLP Model
mlp_model = NeuralCF(num_users, num_items, embedding_dim)
train_model(mlp_model, dataloader)

# Evaluate
print("Hit@10 (MF):", hit_at_k(mf_model, user_ids[:100], list(range(num_items))))
print("Hit@10 (MLP):", hit_at_k(mlp_model, user_ids[:100], list(range(num_items))))

Epoch 1, Loss: 193.4840
Epoch 2, Loss: 189.1285
Epoch 3, Loss: 179.3723
Epoch 4, Loss: 172.2414
Epoch 5, Loss: 166.9371
Epoch 1, Loss: 54.8160
Epoch 2, Loss: 54.2012
Epoch 3, Loss: 53.4265
Epoch 4, Loss: 52.5779
Epoch 5, Loss: 51.2269
Hit@10 (MF): 0.015384615384615385
Hit@10 (MLP): 0.2
